# FG2027 experiments — ViTPose only
Run this notebook on a **Google Colab GPU runtime**. No runs have been executed yet.
Order: setup → inventory/download → audit → smoke → pilot → matrix → summary.
The repository branch must be pushed before the clone cell works. Results/checkpoints go to your Drive.


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import subprocess, json, sys, os
assert 'COLAB_RELEASE_TAG' in os.environ, 'Run in Google Colab'
drive.mount('/content/drive')
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'Select Runtime > Change runtime type > GPU'
REPO = Path('/content/fg2027')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch','FG2027',
        'https://github.com/georgebaher/Sign-Language-Recognition-using-Lightweight-Transformers.git', str(REPO)],check=True)
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-fg2027.txt'],check=True)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True))


## Inspect the supplied Drive folder
This lists files without changing them. Use the actual metadata and ViTPose parquet IDs in the next cell.
Only landmark parquets are used; no MediaPipe, angles, or blendshapes.


In [ ]:
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
service = build('drive','v3')
FOLDER_ID = '1tUoqJawvh0-hSxrIrfU3NUFPcQ8Xhusr'
def inventory(folder, prefix=''):
    page = None
    while True:
        result = service.files().list(q=f"'{folder}' in parents and trashed=false", pageToken=page,
            fields='nextPageToken,files(id,name,mimeType,size)', pageSize=1000,
            supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        for f in sorted(result.get('files',[]),key=lambda f:f['name']):
            name = prefix + f['name']
            print(name, f['id'], f.get('size',''))
            if f['mimeType']=='application/vnd.google-apps.folder':
                inventory(f['id'],name+'/')
        page = result.get('nextPageToken')
        if not page: break
inventory(FOLDER_ID)


In [ ]:
# Fill only after inspecting the inventory; IDs cannot be inferred from names.
FILE_IDS = {
    'wlasl': {'metadata.json':'', 'hand_landmarks.parquet':'', 'pose_landmarks.parquet':'', 'face_landmarks.parquet':''},
    'avasag': {'metadata.json':'', 'hand_landmarks.parquet':'', 'pose_landmarks.parquet':'', 'face_landmarks.parquet':''}
}
assert all(fid for files in FILE_IDS.values() for fid in files.values()), 'Enter the eight verified file IDs'
DATA_ROOT = Path('/content/fg2027_data')
for dataset, files in FILE_IDS.items():
    for filename, fid in files.items():
        dest = DATA_ROOT / dataset / (filename if filename.endswith('.json') else 'vitpose/'+filename)
        dest.parent.mkdir(parents=True,exist_ok=True)
        meta = service.files().get(fileId=fid,fields='id,name,size,md5Checksum',supportsAllDrives=True).execute()
        with dest.with_suffix(dest.suffix+'.tmp').open('wb') as f:
            download = MediaIoBaseDownload(f,service.files().get_media(fileId=fid,supportsAllDrives=True))
            done=False
            while not done: _,done=download.next_chunk()
        temporary=dest.with_suffix(dest.suffix+'.tmp')
        if 'size' in meta: assert temporary.stat().st_size==int(meta['size'])
        if 'md5Checksum' in meta:
            import hashlib
            digest=hashlib.md5()
            with temporary.open('rb') as stream:
                for block in iter(lambda:stream.read(1024*1024),b''): digest.update(block)
            assert digest.hexdigest()==meta['md5Checksum']
        temporary.replace(dest)
config=json.loads(Path('configs/fg2027.json').read_text())
CONFIG=Path('/content/drive/MyDrive/FG2027/config.json')
CONFIG.parent.mkdir(parents=True,exist_ok=True)
if not CONFIG.exists(): CONFIG.write_text(json.dumps(config,indent=2))
print(CONFIG)


## Audit
Stops on missing IDs, overlapping splits, mismatched labels, ambiguous people, missing frame IDs, invalid normalization or missing columns.
Compare counts with the reference paper: WLASL 1442/338/258; AVASAG 2596/849/890. Differences require review, not silent filtering.


In [ ]:
def execute(action, *options):
    subprocess.run([sys.executable,'-m','src.fg2027',action,'--config',str(CONFIG),*options],check=True)
execute('audit')


## Protocol review and smoke tests
Before setting the flag, review `FG2027.md` and both data audit files. Confirm pose indices, missingness and source-clip/signer split provenance. Training uses a constant learning rate with no scheduler or warmup.
Smoke tests run synthetic assertions plus train/validation steps for every configuration. Test scores are not inspected.


In [ ]:
PROTOCOL_REVIEWED = False  # Set True only after resolving the documented decisions/data audit.
assert PROTOCOL_REVIEWED, 'Review FG2027.md and both data_audit.json files first'
config=json.loads(CONFIG.read_text())
config['protocol_reviewed']=True
CONFIG.write_text(json.dumps(config,indent=2))
execute('smoke')


## Pilot (one full run)
Measure time and peak GPU memory before the remaining batch. The pilot is a real matrix run and is reused.
Interruptions resume from the last completed epoch with `--resume`; do not change code/config/data in an existing run directory.


In [ ]:
execute('run','--dataset','wlasl','--model','encoder','--modalities','H','--seed','379','--resume')


## Remaining matrix
66 total runs: 22 configurations × 3 seeds. Run in separate Colab sessions as needed. Completed runs are skipped.
Set the launch flag only after reviewing smoke/pilot output and compute availability.


In [ ]:
LAUNCH_BATCH = False
assert LAUNCH_BATCH, 'Review pilot results and runtime before starting the batch'
execute('run','--resume')


In [ ]:
execute('summarize')
from IPython.display import Markdown, display
display(Markdown((Path(config['output_dir'])/'results.md').read_text()))
